# 01 — Playground MCP : ce qui passe vraiment sur le fil

**Incrément 1.1** — aucune base de données ici, uniquement le protocole.

Objectifs :

1. Écrire un serveur MCP jouet avec le SDK v2 (`MCPServer`).
2. Lui parler **à la main**, en JSON-RPC sur stdin/stdout, dans les deux générations du protocole :
   - **2025-11-25** (handshake `initialize`), encore parlée par la plupart des hosts ;
   - **2026-07-28** (`server/discover`, sans handshake ni session).
3. Comprendre la différence entre une **erreur d'outil** (que le modèle lit) et une **erreur de protocole**.
4. Voir pourquoi `print()` est interdit dans un serveur stdio.
5. Refaire la même chose avec le `Client` du SDK, en sous-processus puis **en mémoire** (le futur pattern de tests).
6. Constater que la docstring et les type hints **deviennent le prompt** de l'outil.


## 0. Préparation

Le serveur jouet est écrit dans un dossier temporaire : il ne pollue pas le repo.
On le lance avec `sys.executable`, c'est-à-dire le Python du `.venv` qui exécute ce notebook.

In [ ]:
import json
import os
import queue
import subprocess
import sys
import tempfile
import threading
from importlib.metadata import version
from pathlib import Path

print("Python :", sys.version.split()[0])
print("SDK mcp :", version("mcp"))

DOSSIER = Path(tempfile.mkdtemp(prefix="playground_mcp_"))
print("Dossier de travail :", DOSSIER)

## 1. Le serveur jouet

Deux outils, choisis pour illustrer les trois issues possibles d'un appel :

- `count_letters` réussit… ou **refuse proprement** avec un `ToolError` (message lisible par le modèle) ;
- `divide` **plante** sur une division par zéro (exception imprévue, message masqué au modèle).

À noter : aucun JSON Schema écrit à la main. Le décorateur `@mcp.tool()` lit les **type hints**
et la **docstring**.

In [ ]:
CODE_SERVEUR_JOUET = """\
\"\"\"Serveur MCP jouet : deux outils, aucune base de données.\"\"\"

from mcp.server import MCPServer
from mcp.server.mcpserver.exceptions import ToolError

mcp = MCPServer("jouet")


@mcp.tool()
def count_letters(word: str, letter: str) -> int:
    \"\"\"Compte le nombre d'occurrences d'une lettre dans un mot.

    Args:
        word: Le mot à analyser.
        letter: La lettre à compter (un seul caractère).
    \"\"\"
    if len(letter) != 1:
        raise ToolError("Le paramètre 'letter' doit contenir exactement un caractère.")
    return word.lower().count(letter.lower())


@mcp.tool()
def divide(a: float, b: float) -> float:
    \"\"\"Divise a par b.\"\"\"
    return a / b


if __name__ == "__main__":
    mcp.run("stdio")
"""

SERVEUR_JOUET = DOSSIER / "serveur_jouet.py"
SERVEUR_JOUET.write_text(CODE_SERVEUR_JOUET, encoding="utf-8")
print(SERVEUR_JOUET.read_text(encoding="utf-8"))

## 2. Un client « à la main »

Ce que fait un host (Claude Desktop, Claude Code…) en stdio, réduit à l'essentiel :

- il lance le serveur en **sous-processus** ;
- il écrit des messages JSON-RPC sur le **stdin** du serveur, **un message par ligne** ;
- il lit les réponses sur le **stdout** du serveur, une par ligne ;
- le **stderr** du serveur est réservé aux logs (ici, redirigé vers un fichier).

Pourquoi un thread de lecture ? `readline()` est bloquant : sans thread, une réponse qui n'arrive
jamais figerait le notebook. Le thread pousse chaque ligne dans une file, qu'on lit avec un timeout.

In [ ]:
class ConnexionBrute:
    """Lance un serveur MCP en sous-processus et dialogue en JSON-RPC brut."""

    def __init__(self, chemin_serveur: Path) -> None:
        self.log_stderr = chemin_serveur.with_suffix(".stderr.log")
        self._fichier_stderr = self.log_stderr.open("w", encoding="utf-8")
        self.proc = subprocess.Popen(
            [sys.executable, str(chemin_serveur)],
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            stderr=self._fichier_stderr,
            text=True,
            encoding="utf-8",
            bufsize=1,
            env=self._environnement(),
        )
        self._lignes: queue.Queue[str] = queue.Queue()
        threading.Thread(target=self._lire_stdout, daemon=True).start()

    @staticmethod
    def _environnement() -> dict[str, str]:
        """Environnement du serveur : sans couleurs forcées, logs lisibles dans un fichier.

        Le kernel Jupyter force les couleurs pour ses sous-processus (FORCE_COLOR) ;
        on les retire pour que le log stderr reste du texte brut.
        """
        env = {k: v for k, v in os.environ.items() if k not in ("FORCE_COLOR", "CLICOLOR_FORCE")}
        env["COLUMNS"] = "120"
        return env

    def _lire_stdout(self) -> None:
        for ligne in self.proc.stdout:
            self._lignes.put(ligne)

    def envoyer(self, message: dict) -> None:
        """Écrit un message JSON-RPC sur une seule ligne du stdin du serveur."""
        ligne = json.dumps(message, ensure_ascii=False)
        print("→", ligne)
        self.proc.stdin.write(ligne + "\n")
        self.proc.stdin.flush()

    def recevoir_brut(self, timeout: float = 5.0) -> str:
        """Lit la prochaine ligne du stdout du serveur, telle quelle."""
        return self._lignes.get(timeout=timeout).rstrip("\n")

    def recevoir(self, timeout: float = 5.0) -> dict:
        """Lit la prochaine ligne et la décode en JSON (affichage indenté)."""
        reponse = json.loads(self.recevoir_brut(timeout))
        print("←", json.dumps(reponse, indent=2, ensure_ascii=False))
        return reponse

    def fermer(self) -> None:
        """Ferme stdin : le serveur voit la fin du flux et s'arrête."""
        self.proc.stdin.close()
        self.proc.wait(timeout=5)
        self._fichier_stderr.close()

## 3. Génération 2025-11-25 : le handshake `initialize`

Séquence classique : `initialize` → `notifications/initialized` → `tools/list` → `tools/call`.

### 3.1 `initialize`
Le client propose une version de protocole et annonce ses capacités ; le serveur répond avec
la version retenue, ses propres capacités et son identité (`serverInfo`).

In [ ]:
SERVEUR_JOUET

In [ ]:
legacy = ConnexionBrute(SERVEUR_JOUET)

legacy.envoyer(
    {
        "jsonrpc": "2.0",
        "id": 1,
        "method": "initialize",
        "params": {
            "protocolVersion": "2025-11-25",
            "capabilities": {},
            "clientInfo": {"name": "playground", "version": "0.1"},
        },
    }
)
legacy.recevoir();

### 3.2 `notifications/initialized`
Message **sans `id`** : en JSON-RPC, c'est une **notification**. Le serveur n'y répond pas.
On ne lit donc rien après l'envoi (une lecture attendrait jusqu'au timeout).

In [ ]:
legacy.envoyer({"jsonrpc": "2.0", "method": "notifications/initialized"})

### 3.3 `tools/list` : ce que le modèle verra

Observe pour chaque outil :

- `description` : **la docstring entière**, section `Args:` comprise, envoyée telle quelle ;
- `inputSchema` : généré depuis les type hints (`str` → `"string"`, `float` → `"number"`) ;
  les paramètres n'ont **pas** de description individuelle (on y revient en partie 7) ;
- `outputSchema` : généré depuis le type de retour (`-> int`), ce qui permet un résultat structuré.

In [ ]:
legacy.envoyer({"jsonrpc": "2.0", "id": 2, "method": "tools/list"})
legacy.recevoir();

### 3.4 `tools/call` : un appel qui réussit
Le résultat arrive deux fois : en `content` (texte, pour le modèle) et en `structuredContent`
(JSON conforme à l'`outputSchema`, pour un programme).

In [ ]:
legacy.envoyer(
    {
        "jsonrpc": "2.0",
        "id": 3,
        "method": "tools/call",
        "params": {"name": "count_letters", "arguments": {"word": "Chinook", "letter": "o"}},
    }
)
legacy.recevoir();

### 3.5 Trois façons d'échouer

| Cas | Ce que reçoit le client | Message visible par le modèle ? |
|---|---|---|
| `ToolError` levé volontairement | `result` avec `isError: true` | **Oui**, message complet |
| Arguments invalides (validation Pydantic) | `result` avec `isError: true` | **Oui**, détail de la validation |
| Exception imprévue (bug, `ZeroDivisionError`…) | `result` avec `isError: true` | **Non** : juste « Error executing tool … » |

Point clé : dans les trois cas, c'est un **`result`**, pas une `error` JSON-RPC.
Pour le protocole, l'appel a réussi ; c'est l'**outil** qui a échoué, et le modèle peut réagir.
C'est pour ça que les garde-fous lèveront des `ToolError` : l'agent saura *pourquoi* sa requête
a été refusée et pourra se corriger.

Rappel : 

def count_letters(word: str, letter: str) -> int:
    """Compte le nombre d'occurrences d'une lettre dans un mot.

    Args:
        word: Le mot à analyser.
        letter: La lettre à compter (un seul caractère).
    """
    if len(letter) != 1:
        raise ToolError("Le paramètre 'letter' doit contenir exactement un caractère.")
    return word.lower().count(letter.lower())

In [ ]:
# Cas 1 : refus explicite par ToolError
legacy.envoyer(
    {
        "jsonrpc": "2.0",
        "id": 4,
        "method": "tools/call",
        "params": {"name": "count_letters", "arguments": {"word": "Chinook", "letter": "oo"}},
    }
)
legacy.recevoir();

In [ ]:
# Cas 2 : argument manquant, rejeté par la validation avant d'exécuter la fonction
legacy.envoyer(
    {
        "jsonrpc": "2.0",
        "id": 5,
        "method": "tools/call",
        "params": {"name": "count_letters", "arguments": {"word": "Chinook"}},
    }
)
legacy.recevoir();

In [ ]:
# Cas 3 : exception imprévue, le détail est masqué au modèle
legacy.envoyer(
    {
        "jsonrpc": "2.0",
        "id": 6,
        "method": "tools/call",
        "params": {"name": "divide", "arguments": {"a": 1, "b": 0}},
    }
)
legacy.recevoir();

### 3.6 Et les détails, où sont-ils ?
Dans le **stderr du serveur** : la traceback complète du `ZeroDivisionError` y est loggée.
Le modèle ne voit rien de l'intérieur du serveur, le développeur voit tout.

In [ ]:
legacy.fermer()
print(legacy.log_stderr.read_text(encoding="utf-8"))

## 4. Génération 2026-07-28 : plus de handshake

Plus d'`initialize`, plus de session. Le client fait un appel de découverte `server/discover`,
puis **chaque requête porte elle-même** la version du protocole et les capacités du client,
dans `params._meta`, sous des clés réservées `io.modelcontextprotocol/...`.

Conséquence : chaque requête se suffit à elle-même. C'est ce qui permettra plus tard, en HTTP,
de répartir les requêtes sur plusieurs instances sans « session collante ».

In [ ]:
META_2026 = {
    "io.modelcontextprotocol/protocolVersion": "2026-07-28",
    "io.modelcontextprotocol/clientCapabilities": {},
}

moderne = ConnexionBrute(SERVEUR_JOUET)
moderne.envoyer(
    {"jsonrpc": "2.0", "id": 1, "method": "server/discover", "params": {"_meta": META_2026}}
)
moderne.recevoir();

In [ ]:
# Appel direct, sans aucun handshake préalable : l'enveloppe _meta suffit.
moderne.envoyer(
    {
        "jsonrpc": "2.0",
        "id": 2,
        "method": "tools/call",
        "params": {
            "name": "count_letters",
            "arguments": {"word": "Chinook", "letter": "o"},
            "_meta": META_2026,
        },
    }
)
moderne.recevoir();

### 4.1 Une vraie erreur de protocole
Même appel, sans `_meta` et sans handshake : le serveur ne sait pas quelle version on parle.
Cette fois, la réponse contient **`error`** (code JSON-RPC `-32602`, paramètres invalides),
et non un `result`. Ce type d'erreur concerne le **client**, pas le modèle.

In [ ]:
moderne.envoyer(
    {
        "jsonrpc": "2.0",
        "id": 3,
        "method": "tools/call",
        "params": {"name": "count_letters", "arguments": {"word": "Chinook", "letter": "o"}},
    }
)
moderne.recevoir()
moderne.fermer()

## 5. Le piège du `print()`

En stdio, **stdout est le canal du protocole**. Deux cas à comparer :

- un `print()` exécuté **au chargement du module**, avant que le serveur tourne : il part sur stdout ;
- un `print()` **dans un outil**, pendant que le serveur tourne : le SDK v2 le détourne vers stderr
  (au mieux, sans garantie).

In [ ]:
CODE_SERVEUR_BAVARD = """\
from mcp.server import MCPServer

print("Démarrage du serveur bavard...")  # piège n°1 : print au chargement du module

mcp = MCPServer("bavard")


@mcp.tool()
def echo(text: str) -> str:
    \"\"\"Renvoie le texte reçu.\"\"\"
    print(f"debug : echo({text!r})")  # piège n°2 : print pendant un appel d'outil
    return text


if __name__ == "__main__":
    mcp.run("stdio")
"""
SERVEUR_BAVARD = DOSSIER / "serveur_bavard.py"
SERVEUR_BAVARD.write_text(CODE_SERVEUR_BAVARD, encoding="utf-8")

bavard = ConnexionBrute(SERVEUR_BAVARD)
bavard.envoyer(
    {
        "jsonrpc": "2.0",
        "id": 1,
        "method": "initialize",
        "params": {
            "protocolVersion": "2025-11-25",
            "capabilities": {},
            "clientInfo": {"name": "playground", "version": "0.1"},
        },
    }
)
# On lit la première ligne SANS la décoder : ce n'est pas du JSON.
print("Première ligne reçue sur stdout :", repr(bavard.recevoir_brut()))
bavard.recevoir();

In [ ]:
bavard.envoyer({"jsonrpc": "2.0", "method": "notifications/initialized"})
bavard.envoyer(
    {
        "jsonrpc": "2.0",
        "id": 2,
        "method": "tools/call",
        "params": {"name": "echo", "arguments": {"text": "salut"}},
    }
)
bavard.recevoir()
bavard.fermer()
print("stderr du serveur :", bavard.log_stderr.read_text(encoding="utf-8"))

Bilan : la ligne `Démarrage du serveur bavard...` est arrivée **au milieu du flux JSON-RPC**.
Le `Client` Python la loggue comme erreur de parsing et continue ; d'autres hosts peuvent
couper la connexion, et le symptôme ressemble alors à « serveur connecté mais sans outils ».
Le `print()` dans l'outil, lui, a été détourné vers stderr.

**Règle du projet :** jamais de `print()` dans le serveur, uniquement `logging` vers stderr.

## 6. La même chose avec le `Client` du SDK

Tout ce qu'on vient de faire à la main, le `Client` le fait pour nous : lancement du sous-processus,
négociation de version (il tente `server/discover`, puis se replie sur `initialize` si le serveur
est ancien), numérotation des `id`, décodage des réponses.

Jupyter exécute déjà une boucle asyncio : on peut utiliser `async with` / `await` directement dans
les cellules.

In [ ]:
from mcp import Client, StdioServerParameters

parametres = StdioServerParameters(command=sys.executable, args=[str(SERVEUR_JOUET)])

async with Client(parametres) as client:
    print("Version négociée :", client.protocol_version)
    print("Serveur :", client.server_info.name if client.server_info else None)

    outils = await client.list_tools()
    print("Outils :", [outil.name for outil in outils.tools])

    resultat = await client.call_tool("count_letters", {"word": "Chinook", "letter": "o"})
    print("is_error :", resultat.is_error, "| structuré :", resultat.structured_content)

    resultat = await client.call_tool("count_letters", {"word": "Chinook", "letter": "oo"})
    print("is_error :", resultat.is_error, "|", resultat.content[0].text)

Note : `call_tool` **ne lève pas d'exception** quand l'outil échoue. Il faut tester `resultat.is_error`.
C'est cohérent avec la partie 3.5 : l'échec d'un outil est un résultat, pas une erreur de protocole.

### 6.1 Forcer l'ancienne génération
`mode="legacy"` force le handshake `initialize`, comme le font encore la plupart des hosts.
Le même serveur sert les deux générations sans configuration.

In [ ]:
async with Client(parametres, mode="legacy") as client:
    print("Version négociée :", client.protocol_version)

### 6.2 En mémoire : le pattern des futurs tests
On peut passer **l'objet serveur lui-même** au `Client` : pas de sous-processus, pas de JSON
sur un pipe, juste des appels Python. C'est rapide et déterministe, idéal pour pytest.

In [ ]:
import importlib.util

spec = importlib.util.spec_from_file_location("serveur_jouet", SERVEUR_JOUET)
module_jouet = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module_jouet)

async with Client(module_jouet.mcp) as client:
    print("Version négociée :", client.protocol_version)
    resultat = await client.call_tool("divide", {"a": 1, "b": 0})
    print("is_error :", resultat.is_error, "|", resultat.content[0].text)

En mémoire, les logs du serveur s'affichent directement dans le notebook (la traceback du
`ZeroDivisionError` ci-dessus). Côté modèle, toujours le message générique.


## 7. La docstring est le prompt

Partie 3.3 : la docstring part **en entier** dans `description`, et les paramètres n'ont pas de
description propre. Pour en donner une, on annote le paramètre avec `Field(description=...)`.

Comparons les deux écritures sur une préfiguration de notre futur `get_schema`.

In [ ]:
from typing import Annotated

from mcp.server import MCPServer
from pydantic import Field

serveur_demo = MCPServer("demo-schema")


@serveur_demo.tool()
def get_schema_docstring(table: str) -> list[dict]:
    """Décrit les colonnes d'une table existante.

    Args:
        table: Nom exact d'une table renvoyée par list_tables.
    """
    return []


@serveur_demo.tool()
def get_schema_field(
    table: Annotated[str, Field(description="Nom exact d'une table renvoyée par list_tables.")],
) -> list[dict]:
    """Décrit les colonnes d'une table existante."""
    return []


async with Client(serveur_demo) as client:
    for outil in (await client.list_tools()).tools:
        print(f"=== {outil.name}")
        print("description :", repr(outil.description))
        print("input_schema :", json.dumps(outil.input_schema, indent=2, ensure_ascii=False))

Les deux transmettent l'information au modèle, mais pas au même endroit :

- **docstring `Args:`** : tout est dans un bloc de texte libre ; simple, lisible dans le code ;
- **`Field(description=...)`** : la description est attachée au paramètre dans le JSON Schema,
  là où un client ou un modèle s'attend à la trouver.




## 8. À retenir

- Un serveur MCP stdio est un **processus** qui lit du JSON-RPC sur stdin et répond sur stdout,
  **une ligne par message**. Il ne connaît pas le LLM.
- Deux générations coexistent : **handshake `initialize`** (2025-11-25) et **requêtes autonomes
  avec `_meta`** (2026-07-28). Le SDK v2 sert les deux.
- Échec d'outil = **`result` avec `isError: true`** (le modèle le lit) ; échec de protocole =
  **`error` JSON-RPC** (le client le gère).
- Seul le message d'un **`ToolError`** atteint le modèle ; toute autre exception est masquée et
  loggée côté serveur.
- **stdout appartient au protocole** : logs via `logging` vers stderr, jamais de `print()`.
- Type hints + docstring = schéma + prompt de l'outil.

## 9. Points supplémentaires

1. Appel d'un outil qui n'existe pas (`"name": "inconnu"`)
2. Appel `tools/call` avec `"b": "zéro"` à `divide` : validation error si tout va bien
3. Pour la v2 si on retire seulement la clé `clientCapabilities` de `_meta` : missing the required envelope key(s)

In [ ]:
spec = importlib.util.spec_from_file_location("serveur_jouet", SERVEUR_JOUET)
module_jouet = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module_jouet)

In [ ]:
async with Client(module_jouet.mcp) as client:
    print("Version négociée :", client.protocol_version)
    print("==========================================")
    print("Test de l'erreur de type pour un argument")
    print("==========================================")
    resultat = await client.call_tool("divide", {"a": 1, "b": "zero"})
    print("is_error :", resultat.is_error, "|", resultat.content[0].text)
    print("==========================================")
    print("Test de l'outil inexistant")
    print("==========================================")
    resultat = await client.call_tool("inconnu", {})
    print("is_error :", resultat.is_error, "|", resultat.content[0].text)

In [ ]:
META_2026 = {
    "io.modelcontextprotocol/protocolVersion": "2026-07-28",
    # "io.modelcontextprotocol/clientCapabilities": {},
}

moderne = ConnexionBrute(SERVEUR_JOUET)
moderne.envoyer(
    {"jsonrpc": "2.0", "id": 1, "method": "server/discover", "params": {"_meta": META_2026}}
)
moderne.recevoir();